In [2]:
# imports de base

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from scipy import stats

# optionnel : pour les graphiques un peu plus jolis si installe
try:
    import seaborn as sns 
    sns.set(style="whitegrid")
except ImportError:
    sns=None

# Affichage plus lisible
pd.set_option("display.max_columns",None)

In [3]:
# chargement  du jeu de donnees e-commerce
# Assurez-vous que le fichier ecommerce_dataset.csv se trouve dans le meme dossier que le notebook

df= pd.read_csv("ecommerce_dataset.csv")
df.head()

,client_id,sexe,âge,revenu_mensuel,segment_age,revenu_cat,produit,canal_achat,montant_panier,panier_frequent,note_satisfaction,retour_produit,date_achat
0,1,Homme,50,3893.70,Mature,Moyen,Sport,Mobile,83.27,Non,2,Non,2023-05-28
1,2,Femme,39,2885.15,Adulte,Moyen,Électronique,Mobile,47.18,Oui,3,Non,2023-01-13
2,3,Homme,44,2369.53,Mature,Moyen,Sport,Mobile,72.28,Oui,4,Non,2023-03-13
3,4,Homme,38,3197.92,Adulte,Moyen,Électronique,Mobile,54.59,Non,3,Non,2023-05-07
4,5,Homme,20,3774.66,Jeune,Moyen,Sport,Mobile,75.30,Non,4,Non,2023-07-04


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   client_id          15000 non-null  int64  
 1   sexe               15000 non-null  str    
 2   âge                15000 non-null  int64  
 3   revenu_mensuel     15000 non-null  float64
 4   segment_age        15000 non-null  str    
 5   revenu_cat         15000 non-null  str    
 6   produit            15000 non-null  str    
 7   canal_achat        15000 non-null  str    
 8   montant_panier     15000 non-null  float64
 9   panier_frequent    15000 non-null  str    
 10  note_satisfaction  15000 non-null  int64  
 11  retour_produit     15000 non-null  str    
 12  date_achat         15000 non-null  str    
dtypes: float64(2), int64(3), str(8)
memory usage: 1.5 MB


### Classification des variables du jeu de données

**Variables Quantitatives** (Données numériques mesurables) :
* **âge** : Variable numérique entière (`int64`) représentant l'âge du client.
* **revenu_mensuel** : Variable numérique décimale (`float64`) mesurant les revenus mensuels.
* **montant_panier** : Variable numérique décimale (`float64`) indiquant la valeur financière de l'achat.
* **note_satisfaction** : Variable numérique entière (`int64`) représentant un score ou une échelle de satisfaction.

**Variables Qualitatives** (Données catégorielles ou textuelles) :
* **sexe** : Variable textuelle (`str`) indiquant le genre du client.
* **segment_age** : Variable textuelle (`str`) regroupant les âges par tranches/catégories.
* **revenu_cat** : Variable textuelle (`str`) regroupant les revenus par tranches/catégories.
* **produit** : Variable textuelle (`str`) désignant l'article ou le type de produit acheté.
* **canal_achat** : Variable textuelle (`str`) indiquant le moyen utilisé pour l'achat (ex: en ligne, en magasin).
* **panier_frequent** : Variable textuelle (`str`) qualifiant les habitudes d'achat.
* **retour_produit** : Variable textuelle (`str`) indiquant si le produit a été retourné ou non (généralement Oui/Non).
* **date_achat** : Variable textuelle (`str`) représentant une date (temporelle), stockée ici sous forme de chaîne de caractères.

*Note : La variable `client_id` (`int64`) est un identifiant unique. Bien qu'elle soit stockée sous forme de nombre entier, elle n'a pas de sens mathématique (on ne peut pas l'additionner ou en faire une moyenne). Elle est donc considérée comme une variable qualitative nominale.*


### 📌 Synthèse de l'Analyse Univariée (Variables Quantitatives)

L'analyse univariée étudie chaque variable séparément selon **trois grands axes complémentaires** :

#### 📊 1. Les Statistiques Descriptives (Indicateurs Numériques)
* **Mesures de tendance centrale** (Le "centre" des données) :
  * **Moyenne** : Valeur moyenne globale (sensible aux extrêmes).
  * **Médiane** : Valeur pivot séparant les données en deux groupes égaux (50% au-dessus, 50% en dessous).
  * **Mode** : Valeur ou classe la plus fréquente.
* **Mesures de dispersion** (L'étalement des données) :
  * **Étendue** : Écart total entre le Maximum et le Minimum.
  * **Variance & Écart-type** : Mesure de l'écart moyen par rapport à la moyenne.
  * **Quartiles (Q1, Q2, Q3) & IQR (Écart Interquartile)** : Division des données en quatre quarts (l'IQR contient les 50% centraux).
* **Mesures de forme** (L'allure de la distribution) :
  * **Asymétrie (Skewness)** : Inclinaison de la courbe à gauche ou à droite.
  * **Aplatissement (Kurtosis)** : Concentration des données ou épaisseur des queues de distribution.

#### 📈 2. Les Visualisations Graphiques
* **L'Histogramme** : Permet d'observer la distribution des fréquences (courbe en cloche, loi normale, distribution bimodale ou asymétrique).
* **La Boîte à moustaches (Boxplot)** : Offre une vue synthétique de la médiane, des quartiles, de la dispersion et de la symétrie.

#### 🔍 3. La Détection des Valeurs Aberrantes (Outliers)
* **Identification** : Repérage des données extrêmes ou anormales qui faussent les analyses.
* **Règle de l'IQR** : Exclusion ou analyse spécifique des points situés en dehors des limites `[Q1 - 1.5 * IQR]` et `[Q3 + 1.5 * IQR]` (visibles aux extrémités du Boxplot).


In [5]:
# on extrait la variable la variables quantitative  a analyser 
variable ="montant_panier"
serie=df[variable]
serie.describe()

count    15000.000000
mean        93.357107
std         66.260809
min         10.320000
25%         48.310000
50%         73.530000
75%        117.295000
max        929.970000
Name: montant_panier, dtype: float64

### 📐 Les Mesures de Tendance Centrale

Les mesures de tendance centrale résument une distribution de données en identifiant une valeur unique qui représente le "centre" ou le cœur du jeu de données.

#### 1. La Moyenne ($\bar{x}$)
La moyenne arithmétique est la somme de toutes les valeurs divisée par le nombre total d'observations. Elle est très sensible aux valeurs extrêmes (outliers).

*   **Formule pour une population ou un échantillon non groupé :**

$$ \bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i = \frac{x_1 + x_2 + \dots + x_n}{n} $$

*   *Où :*
    *   $n$ = nombre total d'observations (taille de l'échantillon).
    *   $x_i$ = valeur de la $i$-ème observation.
    *   $\sum$ = symbole de sommation.

#### 2. La Médiane ($Me$)
La médiane est la valeur pivot qui partage la série statistique ordonnée (du plus petit au plus grand) en deux groupes de même effectif : 50 % des valeurs sont inférieures ou égales à la médiane et 50 % sont supérieures ou égales. Elle est robuste face aux valeurs aberrantes.

Pour la calculer, on commence par trier les données par ordre croissant :

*   **Si le nombre d'observations $n$ est impair :**
    La médiane est la valeur située exactement à la position $\frac{n + 1}{2}$.
    
$$ Me = x_{\left(\frac{n + 1}{2}\right)} $$

*   **Si le nombre d'observations $n$ est pair :**
    La médiane est la moyenne des deux valeurs centrales situées aux positions $\frac{n}{2}$ et $\frac{n}{2} + 1$.
    
$$ Me = \frac{x_{\left(\frac{n}{2}\right)} + x_{\left(\frac{n}{2} + 1\right)}}{2} $$

#### 3. Le Mode ($Mo$)
Le mode est la valeur (ou la classe de valeurs) qui apparaît le plus fréquemment dans le jeu de données. Une distribution peut n'avoir aucun mode, être unimodale (un seul mode) ou multimodale (plusieurs modes).

*   **Formule :**
    Il n'y a pas de formule algébrique directe. On identifie la valeur $x_i$ qui maximise la fonction d'effectif (ou de fréquence) $n_i$ :

$$ Mo = \{ x_i \ | \ n_i = \max(n_1, n_2, \dots, n_k) \} $$


In [6]:
# calculs en python 

mean_value= serie.mean()
median_value= serie.median()
mode_value= serie.mode() # peut retourner plusieurs valeurs

print(f"Moyenne: {mean_value:.2f}")
print(f"Mediane: {median_value:.2f}")
print("Mode(s) :")
print(mode_value)

Moyenne: 93.36
Mediane: 73.53
Mode(s) :
0    62.12
Name: montant_panier, dtype: float64


### 📝 Interprétation des Mesures de Tendance Centrale

À partir des résultats obtenus, nous pouvons tirer des conclusions importantes sur la structure et la forme de vos données :

*   **Moyenne (93.36) vs Médiane (73.53) :** 
    La moyenne est nettement supérieure à la médiane. Cela indique que vos données sont **fortement asymétriques à droite (asymétrie positive)**. La moyenne est tirée vers le haut par la présence de valeurs particulièrement élevées (par exemple, des clients avec un `montant_panier` très important).
*   **Médiane (73.53) :** 
    C'est l'indicateur le plus représentatif du comportement "typique". Elle nous montre que **50 % de vos observations ont une valeur inférieure ou égale à 73.53**, et les autres 50 % ont une valeur supérieure. Contrairement à la moyenne, elle n'est pas faussée par les extrêmes.
*   **Mode (62.12) :** 
    La valeur **62.12 est la valeur la plus fréquente** (le pic de votre distribution). Elle se situe en dessous de la médiane et de la moyenne, ce qui confirme que la majorité de votre population se concentre autour de valeurs plus faibles, tandis qu'une minorité s'étend vers des valeurs très hautes.

#### 📈 Synthèse de la Forme de la Distribution

Puisque nous avons l'ordre suivant :
$$ Mode (62.12) < Médiane (73.53) < Moyenne (93.36) $$

Votre distribution possède une longue "queue" vers les valeurs positives. Si l'on prend l'exemple de la variable `montant_panier`, cela signifie que **la plupart des clients dépensent entre 60 et 75 unités**, mais que quelques très gros acheteurs (outliers) font grimper la moyenne générale à 93.36.


### L'étendue
**Définition :** C'est la mesure de dispersion la plus simple. Elle représente la différence brute entre la valeur la plus élevée et la valeur la plus basse d'une série. Elle est très sensible aux valeurs aberrantes et ne dit rien sur la répartition des données entre les deux extrêmes.
$$E = X_{max} - X_{min}$$

### Variance d'une population
**Définition :** Elle mesure la dispersion globale des données autour de la moyenne. C'est la moyenne des carrés des écarts à la moyenne. Son unité est le carré de l'unité des données initiales, ce qui rend son interprétation concrète difficile.
$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i-\mu)^2$$

### Variance d'un échantillon (corrigée)
**Définition :** Utilisée lorsque l'on travaille sur un échantillon pour estimer la variance de la population totale. On divise par $n-1$ (au lieu de $n$) pour corriger un biais mathématique qui tend à sous-estimer la vraie variabilité.
$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2$$

### Écart-type d'une population
**Définition :** C'est la racine carrée de la variance. L'écart-type exprime la dispersion dans la même unité de mesure que les données d'origine (ex: en mètres, en FCFA), ce qui permet de quantifier l'éloignement "habituel" des valeurs par rapport à la moyenne.
$$\sigma = \sqrt{\sigma^2}$$

### Écart-type d'un échantillon
**Définition :** C'est l'estimation de l'écart-type de la population à partir des données de l'échantillon, calculée à partir de la variance corrigée.
$$s = \sqrt{s^2}$$

### Coefficient de variation (Population & Échantillon)
**Définition :** C'est une mesure de dispersion relative exprimée en pourcentage. Il normalise l'écart-type en le divisant par la moyenne. Cela permet de comparer directement la variabilité de deux séries statistiques différentes, même si leurs moyennes sont très éloignées ou si leurs unités de mesure ne sont pas les mêmes.
$$CV_{pop} = \frac{\sigma}{\mu}\times 100$$
$$CV_{éch} = \frac{s}{\bar{x}}\times 100$$


In [7]:
# Calcul de la variance, de l'écart-type, de l'étendue et du CV

variance = serie.var(ddof=1)       # Variance empirique corrigée
std_dev = serie.std(ddof=1)        # Écart-type
vmin = serie.min()                # Valeur minimale
vmax = serie.max()                # Valeur maximale
etendue = vmax - vmin             # Étendue

mean_value = serie.mean()         # Moyenne

cv = (std_dev / mean_value) * 100  # Coefficient de variation en %

# Affichage des résultats
print(f"Variance : {variance:.2f}")
print(f"Écart-type : {std_dev:.2f}")
print(f"Minimum : {vmin:.2f}")
print(f"Maximum : {vmax:.2f}")
print(f"Étendue : {etendue:.2f}")
print(f"Moyenne : {mean_value:.2f}")
print(f"Coefficient de variation : {cv:.2f}%")

Variance : 4390.49
Écart-type : 66.26
Minimum : 10.32
Maximum : 929.97
Étendue : 919.65
Moyenne : 93.36
Coefficient de variation : 70.98%
